# Fake News Detection — End-to-End Pipeline

Companion notebook to the research project:

> **Systematic Review of Fake News, Propaganda, and Disinformation: Examining Authors, Content, and Social Impact** — using Machine Learning and Deep Learning algorithms for social media detection.

### References
1. Plikynas, D., Rizgeliene, I., & Korvel, G. (2025). *Systematic Review of Fake News, Propaganda, and Disinformation.* **IEEE Access 13:17583–17629.** [DOI: 10.1109/ACCESS.2025.3530688](https://doi.org/10.1109/ACCESS.2025.3530688)
2. Devarajan, G. G., et al. (2023). *AI-Assisted Deep NLP-Based Approach for Prediction of Fake News From Social Media Users.* **IEEE Access.**
3. JOAIR (jouair.com) — Indonesian-context methodology reference.

### Notebook phases
1. **Setup & data loading** — from scraper CSVs or public labelled datasets.
2. **Preprocessing** — Indonesian + English text normalisation.
3. **Model implementation** — classical ML baselines + Hugging Face transformers.
4. **Evaluation** — accuracy / precision / recall / F1 / confusion matrix.
5. **Cross-reference** — verify against Komdigi, Kompas, CNN ID, CNBC ID, Tempo, TribrataNews.
6. **User-facing inference** — predict on a caption or on an uploaded image (OCR).

## Phase 1 — Setup & data loading

In [ ]:
# --- Environment bootstrap (works both locally AND on Google Colab) ---
import sys, os, json, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Running on:', 'Google Colab' if IN_COLAB else 'local Jupyter')

def _has(mod):
    try:
        __import__(mod); return True
    except ImportError:
        return False

def _find_pkg_dir():
    here = Path.cwd()
    for candidate in [
        here, here.parent,
        Path('/content/social-media-scrapper/fake_news_detection'),
        Path('/content/fake_news_detection'),
        Path('/content/drive/MyDrive/social-media-scrapper/fake_news_detection'),
    ]:
        if (candidate / 'src' / 'preprocessing.py').exists():
            return candidate
    return None

PKG_DIR = _find_pkg_dir()

# ---- COLAB FALLBACK: clone the repo if we didn't find it ----
if IN_COLAB and PKG_DIR is None:
    REPO_URL = os.environ.get('SMS_REPO_URL', '').strip()
    if REPO_URL:
        target = Path('/content/social-media-scrapper')
        if not target.exists():
            print(f'Cloning {REPO_URL} ...')
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        PKG_DIR = _find_pkg_dir()
    else:
        raise RuntimeError(
            'fake_news_detection/ not found on Colab. Two options:\n'
            '  (a) Upload: click the folder icon in the left pane -> Upload -> '
            'upload the whole fake_news_detection/ folder (as a zip, then !unzip).\n'
            '  (b) Clone: set  os.environ["SMS_REPO_URL"] = "https://github.com/<you>/<repo>.git"  '
            'in a cell above and re-run.'
        )

assert PKG_DIR is not None and (PKG_DIR / 'src' / 'preprocessing.py').exists(), (
    f'Could not locate fake_news_detection/. cwd={Path.cwd()}'
)

sys.path.insert(0, str(PKG_DIR))
os.chdir(PKG_DIR / 'notebook')
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = PKG_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

# ---- install ML deps (Colab / fresh envs) ----
_needed = [
    ('sklearn', 'scikit-learn'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('xgboost', 'xgboost'),
    ('transformers', 'transformers'),
    ('torch', 'torch'),
    ('bs4', 'beautifulsoup4'),
    ('lxml', 'lxml'),
    ('requests', 'requests'),
]
_to_install = [pkg for mod, pkg in _needed if not _has(mod)]
if _to_install:
    print('Installing:', ', '.join(_to_install))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_to_install])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

print('\nPython   :', sys.version.split()[0])
print('Package  :', PKG_DIR)
print('Data dir :', DATA_DIR)

In [ ]:
# --- OPTION A: load a CSV produced by the scraper (main.py) ---
# Any of instagram_result.csv / tiktok_result.csv / twitter_result.csv will work.
# The 'caption' column is what we'll analyse.

SCRAPER_CSV = NOTEBOOK_DIR.parent.parent / 'instagram_result.csv'  # adjust as needed
if SCRAPER_CSV.exists():
    scraped = pd.read_csv(SCRAPER_CSV)
    print(f'Loaded {len(scraped)} scraped rows from {SCRAPER_CSV.name}')
    display(scraped[['platform','username','caption']].head())
else:
    print(f'(No scraped CSV found at {SCRAPER_CSV} — that is fine for training.)')
    scraped = None

In [ ]:
# --- OPTION B: load a labelled dataset for TRAINING the model ---
# Drop any labelled CSV with two columns (text, label) into data/train.csv.
# Recommended public datasets:
#   - Kaggle: 'Fake and Real News' (English)
#   - HF: Pulk17/Fake-News-Detection-dataset
#   - Indonesian: 'Muhammad-Falah-Zaim/Indonesian-Hoax-News-Detection'

TRAIN_CSV = DATA_DIR / 'train.csv'
if TRAIN_CSV.exists():
    df = pd.read_csv(TRAIN_CSV)
    print(f'Loaded training set: {len(df)} rows')
else:
    # Tiny synthetic demo so the notebook runs end-to-end without a dataset
    df = pd.DataFrame({
        'text': [
            'Presiden menandatangani undang-undang baru hari ini di Jakarta.',
            'Vaksin COVID-19 mengandung microchip pelacak, kata akun anonim.',
            'BMKG memperkirakan hujan lebat di wilayah Jabodetabek besok.',
            'Menkominfo mengumumkan program literasi digital nasional.',
            'Air hujan sekarang beracun karena chemtrail, jangan keluar rumah!',
            'Polri tangkap jaringan penipuan daring lintas provinsi.',
            'Foto viral tsunami raksasa di Bali TERNYATA hoaks lama 2011.',
            'Pemerintah rilis bansos baru untuk keluarga miskin.',
            'Klik link ini untuk bansos Rp 5 juta langsung cair hari ini!!!',
            'Kompas melaporkan pertumbuhan ekonomi kuartal III sebesar 5.1%.',
        ],
        'label': [0,1,0,0,1,0,1,0,1,0]   # 0=real, 1=fake
    })
    print('Using built-in 10-row synthetic demo. Replace data/train.csv with a real dataset.')

df.head()

## Phase 2 — Preprocessing

Indonesian-aware text normalisation: strip URLs, mentions, emojis, punctuation. Optionally remove stopwords / stem with Sastrawi.

In [ ]:
from src.preprocessing import TextCleaner

cleaner = TextCleaner(
    language='id',
    strip_urls=True,
    strip_mentions=True,
    strip_hashtags=False,
    strip_emoji=True,
    strip_punct=True,
    remove_stopwords=True,
    stem=False,          # flip to True if Sastrawi is installed
)

df['text_clean'] = df['text'].apply(cleaner)
df[['text','text_clean','label']].head()

## Phase 3 — Model implementation

### 3a. Classical ML baselines (TF-IDF + LR / SVM / RF / XGB)

In [ ]:
from sklearn.model_selection import train_test_split
from src.models import ClassicalMLModel
from src.metrics import compute_metrics, print_metrics, confusion_plot, compare_models

X = df['text_clean'].tolist()
y = df['label'].tolist()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y if len(set(y)) > 1 else None,
)

classical_results = {}
for kind in ['logreg','svm','rf']:
    m = ClassicalMLModel(kind=kind).fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    y_proba = m.predict_proba(X_te)[:,1] if len(set(y)) == 2 else None
    classical_results[kind] = compute_metrics(y_te, y_pred, y_proba)
    print(f'\n=== {kind.upper()} ===')
    print_metrics(classical_results[kind])

compare_models(classical_results)

### 3b. Hugging Face transformer (zero-shot inference)

We use a pre-trained fake-news model from the Hugging Face hub. First run downloads the weights (~500 MB). For production, fine-tune on your labelled dataset via `HuggingFaceClassifier.fine_tune(...)`.

In [ ]:
from src.models import HuggingFaceClassifier, AVAILABLE_HF_MODELS

# Pick a model — options:
#   'roberta-fake-news', 'bert-fake-news', 'indobert',
#   'indobert-sentiment', 'xlm-roberta', 'mbert'
HF_MODEL = AVAILABLE_HF_MODELS['roberta-fake-news']
print('Using:', HF_MODEL)

hf = HuggingFaceClassifier(model_id=HF_MODEL)

# Inference on the test split
hf_preds = hf.predict(X_te)
hf_proba = hf.predict_proba(X_te)
print('Pipeline label map:', hf._label_map)
print('Sample preds:', hf_preds[:5])

In [ ]:
# Map the HF pipeline labels to our {0=real, 1=fake} space before scoring.
# Most 'fake-news' HF models use labels like 'REAL'/'FAKE' or 'LABEL_0'/'LABEL_1'.
def normalise_label(lbl: str) -> int:
    s = lbl.upper()
    if 'FAKE' in s or s.endswith('_1') or s == '1':
        return 1
    return 0

hf_y_pred = [normalise_label(l) for l in hf_preds]
hf_metrics = compute_metrics(y_te, hf_y_pred, y_proba=hf_proba[:,1] if hf_proba.shape[1] >= 2 else None)
print('\n=== Hugging Face (zero-shot) ===')
print_metrics(hf_metrics)

## Phase 4 — Evaluation summary

Aggregate all models side-by-side and visualise the winner's confusion matrix.

In [ ]:
all_results = {**classical_results, 'huggingface': hf_metrics}
leaderboard = compare_models(all_results)
leaderboard

In [ ]:
best_name = leaderboard.iloc[0]['model']
print(f'Best model by F1: {best_name}')

fig, ax = plt.subplots(figsize=(5,4))
confusion_plot(all_results[best_name], labels=['Real','Fake'], title=f'Confusion — {best_name}', ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Persist metrics for the Streamlit app to load
for name, m in all_results.items():
    out_path = DATA_DIR / f'metrics_{name}.json'
    payload = {k: v for k, v in m.items() if k != 'classification_report'}
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    print('saved', out_path.name)

## Phase 5 — Cross-reference against credible sources

Build a corpus from the six verified Indonesian sources and match any caption against it.

Sources: **komdigi.go.id**, **tribratanews.jabar.polri.go.id**, **cnbcindonesia.com**, **kompas.com**, **cnnindonesia.com**, **tempo.co**.

In [ ]:
from src.cross_reference import CredibleSourceMatcher, VERIFIED_SOURCES

matcher = CredibleSourceMatcher(
    max_articles_per_source=30,
    use_embeddings=False,   # flip to True to use sentence-transformers (slower, better)
)
n = matcher.build_corpus()
print(f'Indexed {n} articles across {len(VERIFIED_SOURCES)} sources.')

In [ ]:
query = 'Klik link ini untuk bansos Rp 5 juta langsung cair hari ini!!!'
hits = matcher.match(cleaner(query), top_k=5)

for h in hits:
    print(f"{h.similarity:.3f}  [{h.article.source_name}] {h.article.title[:90]}")
    print(f"        {h.article.url}")

## Phase 6 — User-facing inference

Predict on an arbitrary caption OR an uploaded image.

In [ ]:
def analyse(text: str | None = None, image_path: str | None = None, top_k: int = 5):
    if image_path:
        from src.ocr import extract_text_from_image
        text = extract_text_from_image(image_path)
        print(f'[OCR] {text}\n')

    assert text, 'provide text= or image_path='
    cleaned = cleaner(text)

    proba = hf.predict_proba([cleaned])[0]
    lbl_map = hf._label_map or {i: str(i) for i in range(len(proba))}
    top = int(np.argmax(proba))
    print(f'Prediction : {lbl_map[top]}   (confidence {proba[top]*100:.1f}%)')
    for i, p in enumerate(proba):
        print(f'  {lbl_map[i]:<20} {p:.4f}')

    print('\nTop verified-source matches:')
    for h in matcher.match(cleaned, top_k=top_k):
        print(f'  {h.similarity:.3f}  [{h.article.source_name}] {h.article.title[:80]}')
    return {'text': text, 'cleaned': cleaned, 'pred': lbl_map[top], 'confidence': float(proba[top])}

# Demo
analyse(text='Vaksin COVID-19 mengandung microchip pelacak menurut akun anonim.')

In [ ]:
# Image example — point at any screenshot file:
# analyse(image_path=r'C:\Users\luthfi\Pictures\Screenshots\some-caption.png')

## Next steps

1. **Replace the synthetic demo dataset** in `data/train.csv` with a real labelled corpus (e.g., Kaggle Fake News, Indonesian-Hoax-News-Detection).
2. **Fine-tune the Hugging Face model** on that dataset using `HuggingFaceClassifier.fine_tune(...)` — this is where the biggest F1 gains come from.
3. **Batch-analyse scraper output** — load any CSV from `../*_result.csv`, run `hf.predict(df['caption'])`, and write back a `prediction` column.
4. **Launch the Streamlit app** — `streamlit run ../app/streamlit_app.py` — for the end-user upload interface described in the research protocol.